<a href="https://colab.research.google.com/github/FernandaChacara/project-template/blob/main/Exemplo_Teste1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as sts
import zipfile
import scikit_posthocs as sp
import statsmodels.stats as stm
from statsmodels.graphics.gofplots import qqplot
import math

##### Q1 - Explore the dataset, including the number of records, variable names, and the presence of missing values.

número de linhas e colunas (df.shape)
nomes das variáveis (df.columns)
presença ou ausência de missing values (df.isna().sum())
tipos de dados (df.info())

In [ ]:
df = pd.read_csv(
    '../Exemplos/greenhouse_gas_inventory_data.zip',
    compression='zip',
    sep=','
)

# estrutura geral
df.shape
df.columns

# tipos de dados
df.info()

# valores em falta
df.isna().sum()

# visão rápida
df.head()
df.describe()

##### Q2 - Plot the mean GHG emission values of CO<sub>2</sub> per country. Include an error bar.
(NOTE:The CO<sub>2</sub> emission is coded in the variable named `category` as `carbon_dioxide_co2_emissions_without_land_use_land_use_change_and_forestry_lulucf_in_kilotonne_co2_equivalent`).


In [ ]:
co2 = df[df['category'] ==
         'carbon_dioxide_co2_emissions_without_land_use_land_use_change_and_forestry_lulucf_in_kilotonne_co2_equivalent']

summary = co2.groupby('country_or_area')['value'].agg(['mean', 'std']).reset_index()

plt.figure(figsize=(12,6))

plt.bar(
    summary['country_or_area'],
    summary['mean'],
    yerr=summary['std'],
    capsize=3
)

plt.xticks(rotation=90)
plt.ylabel('Mean CO₂ emissions (kt CO₂ eq)')
plt.title('Mean CO₂ emissions per country with variability')
plt.tight_layout()
plt.show()

##### Q3 - Compute the mean, minimum, maximum, quartiles and standard deviation of GHG emmission values of CO<sub>2</sub> (carbon_dioxide_co2_emissions_without_land_use_land_use_change_and_forestry_lulucf_in_kilotonne_co2_equivalent) in 1990 and 2014, and separately for Portugal and Greece.

In [ ]:
countries = ['Portugal', 'Greece']
years = [1990, 2014]

co2_subset = co2[co2['country_or_area'].isin(countries) &
                 co2['year'].isin(years)]

def stats(x):
    return {
        'mean': np.mean(x),
        'min': np.min(x),
        'max': np.max(x),
        'q1': np.quantile(x, 0.25),
        'median': np.median(x),
        'q3': np.quantile(x, 0.75),
        'std': np.std(x, ddof=1)
    }

result = co2_subset.groupby(['country_or_area','year'])['value'].apply(stats)
result

##### Q4 - Produce a plot that shows differences/similarities between the two countries in the GHG annual emission values.

In [ ]:
plt.figure(figsize=(10,5))

sns.lineplot(
    data=co2[co2['country_or_area'].isin(['Portugal','Greece'])],
    x='year',
    y='value',
    hue='country_or_area'
)

plt.title('CO₂ emissions over time: Portugal vs Greece')
plt.ylabel('kt CO₂ eq')
plt.show()

sns.boxplot(
    data=co2[co2['country_or_area'].isin(['Portugal','Greece'])],
    x='country_or_area',
    y='value'
)
plt.title('Distribution of CO₂ emissions')
plt.show()

##### Q5 - Carefully analyse the central tendency estimates of GHG emmisions you computed on Q3. Is the relationship between the mean and the median the same between countries? Comment your response taking into account the plot produced in Q4.

A relação entre média e mediana permite avaliar assimetria da distribuição. Se a média for superior à mediana, a distribuição tende a ser assimétrica à direita, indicando a presença de anos com emissões excecionalmente elevadas. Se forem semelhantes, a distribuição tende a ser aproximadamente simétrica.

No caso de Portugal e Grécia, a comparação com o gráfico temporal sugere que as emissões não são estáveis ao longo do tempo, apresentando períodos de maior intensidade seguidos de redução. Assim, diferenças entre média e mediana refletem essa variabilidade temporal e possíveis outliers associados a picos de emissão, sendo expectável que a média seja mais sensível a esses valores extremos do que a mediana.

##### Q6 - Assuming that annual CO<sub>2</sub> GHG emission values follow a normal distribution (even if not necessarily true), which statistical test would be appropriate to compare the values between the Portugal and Greece, given the data characteristics? Define the null hypothesis of the test you want to run, the critical *p-value*, run the test and justify briefly your choice.

In [ ]:
pt = co2[co2['country_or_area']=='Portugal']['value']
gr = co2[co2['country_or_area']=='Greece']['value']

sts.ttest_ind(pt, gr, equal_var=False)

##### Q7 - Run an analysis that tests the following null Hypothesis: *H0 - the annual GH Gas Emmision values of Portugal and Greece between 1990 and 2014 follows the same probability distribution*. Use a critical *p-value* of 0.05.

In [ ]:
sts.mannwhitneyu(pt, gr, alternative='two-sided')

##### Q8 - Now imagine you wanted to test if GHG emmission values of CO<sub>2</sub>  changed over the years 1990, 1995, 2000, 2005 and 2010 considering all countries with data. Specify the null and alternative hypothesis of the performed test, the critical alpha used as well as the effect sizes. Check also if the direction of changes in the global emmissions were consistent between the time periods.

In [ ]:
sts.f_oneway(
    co2[co2['year']==1990]['value'],
    co2[co2['year']==1995]['value'],
    co2[co2['year']==2000]['value'],
    co2[co2['year']==2005]['value'],
    co2[co2['year']==2010]['value']
)

SS_between / SS_total